# Integração do pipeline UCE

Este notebook funciona como um auditor do projeto. Ele não recalcula todas as
etapas pesadas; verifica os arquivos produzidos e resume o caminho completo:

SRA → FASTQ → QC/trimming → SPAdes → contigs → PHYLUCE → UCE FASTA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import gzip, csv, statistics

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
OUT = ROOT / "08_integracao"
OUT.mkdir(parents=True, exist_ok=True)

paths = {
    "R1_raw": ROOT/"03_sra_fastq/SRR15736591_1.fastq.gz",
    "R2_raw": ROOT/"03_sra_fastq/SRR15736591_2.fastq.gz",
    "R1_trim": ROOT/"04_qc_trimming/SRR15736591_R1_paired.fastq.gz",
    "R2_trim": ROOT/"04_qc_trimming/SRR15736591_R2_paired.fastq.gz",
    "contigs": ROOT/"05_spades/SRR15736591/contigs_min200.fasta",
    "uce_db": ROOT/"06_uce_match/uce-search-results/probe.matches.sqlite",
    "uce_fasta": ROOT/"07_uce_extract/SRR15736591-incomplete.fasta",
}
for k,p in paths.items():
    print(f"{k:10s}", "OK" if p.exists() else "AUSENTE", p)

## 1. Funções de contagem

In [ ]:
def nreads(path):
    if not path.exists():
        return None
    with gzip.open(path, "rt") as f:
        return sum(1 for _ in f)//4

def fasta_lengths(path):
    if not path.exists():
        return []
    lengths, seq = [], []
    with open(path) as f:
        for line in f:
            line=line.strip()
            if line.startswith(">"):
                if seq:
                    lengths.append(len("".join(seq)))
                seq=[]
            else:
                seq.append(line)
        if seq:
            lengths.append(len("".join(seq)))
    return lengths

def n50(lengths):
    if not lengths:
        return None
    half = sum(lengths)/2
    acc = 0
    for L in sorted(lengths, reverse=True):
        acc += L
        if acc >= half:
            return L

## 2. Métricas

In [ ]:
raw = nreads(paths["R1_raw"])
trim = nreads(paths["R1_trim"])

contig_lengths = fasta_lengths(paths["contigs"])
uce_lengths = fasta_lengths(paths["uce_fasta"])

metrics = {
    "run": "SRR15736591",
    "reads_R1_raw": raw,
    "reads_R1_paired_trimmed": trim,
    "retencao_pct": round(trim/raw*100,2) if raw and trim is not None else None,
    "n_contigs_min200": len(contig_lengths),
    "assembly_total_bp": sum(contig_lengths) if contig_lengths else None,
    "assembly_max_contig": max(contig_lengths) if contig_lengths else None,
    "assembly_N50": n50(contig_lengths),
    "n_uce_records": len(uce_lengths),
    "uce_mean_length": round(sum(uce_lengths)/len(uce_lengths),1) if uce_lengths else None,
}

metrics

## 3. Criar tabela-resumo

In [ ]:
import pandas as pd
summary = pd.DataFrame([metrics])
summary.T

## 4. Salvar CSV final

In [ ]:
csv_path = OUT / "resumo_pipeline_SRR15736591.csv"
summary.to_csv(csv_path, index=False)
print(csv_path)

## 5. Reconstruir o fluxo

Para cada seta abaixo, explique qual arquivo entra e qual arquivo sai:

**SRA → FASTQ → FastQC/MultiQC → Trimmomatic → SPAdes → contigs → PHYLUCE → loci UCE**

Perguntas finais:

1. Em qual etapa deixamos de trabalhar com reads e passamos a trabalhar com contigs?
2. Em qual etapa as sondas UCE passam a participar da análise?
3. O que mudaria se utilizássemos todos os reads do SRA?
4. O que seria necessário para construir uma matriz de occupancy com várias amostras?
5. Quais informações precisamos registrar para outra pessoa reproduzir a análise?

## Encerramento

O objetivo desta sequência não foi reproduzir integralmente Ciaccio et al. (2022),
mas aprender a lógica de um pipeline real utilizando dados públicos relacionados
ao estudo.

A principal habilidade é conseguir explicar **o que cada etapa recebe, transforma
e produz**.